# Fine-Tuning Qwen3 on HOMO-LUMO Data with Tinker: SFT + GRPO

This notebook walks you through fine-tuning a **Qwen3** model on the `Alan123/homo-lumo-grpo` dataset using the **Tinker API** from Thinking Machines Lab.

We'll cover two training stages:
1. **Supervised Fine-Tuning (SFT)** — teach the model the format and basic chemistry reasoning
2. **GRPO (Group Relative Policy Optimization)** — use reinforcement learning with a reward signal to improve accuracy

**What is Tinker?** Tinker is a training API that handles distributed GPU infrastructure so you can focus on data and algorithms. You write a simple Python loop on your CPU machine; Tinker runs the actual training on GPU clusters behind the scenes using LoRA.

**Prerequisites:**
- A Tinker API key (sign up at [tinker-console.thinkingmachines.ai](https://tinker-console.thinkingmachines.ai))
- A HuggingFace account with access to the dataset

**References:**
- [Tinker Docs](https://tinker-docs.thinkingmachines.ai/)
- [Tinker Cookbook](https://github.com/thinking-machines-lab/tinker-cookbook)
- [Loss Functions](https://tinker-docs.thinkingmachines.ai/losses)

## 1. Installation & Setup

In [ ]:
# Install Tinker SDK and dependencies
%pip install -q tinker transformers torch datasets numpy

In [ ]:
# 安装/更新 widgets 插件
%pip install -q -U ipywidgets jupyterlab_widgets

# 如果你是在本地 VS Code 或 Jupyter Lab 运行，建议安装完后重启一下内核 (Restart Kernel)

In [ ]:
# (Optional) Install the Tinker Cookbook for higher-level abstractions
!pip install -q git+https://github.com/thinking-machines-lab/tinker-cookbook.git

In [1]:
import os
from getpass import getpass

# Set your Tinker API key
if "TINKER_API_KEY" not in os.environ:
    os.environ["TINKER_API_KEY"] = getpass("Enter your Tinker API key: ")

# (Optional) HuggingFace token if the dataset is private/gated
if "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = getpass("Enter your HuggingFace token (or press Enter to skip): ")

In [2]:
import tinker
from tinker import types
import numpy as np
import json
import time

# Verify connection to Tinker
service_client = tinker.ServiceClient()
capabilities = service_client.get_server_capabilities()
print("Connected to Tinker! Available models:")
for m in capabilities.supported_models:
    if "Qwen3" in m.model_name:
        print(f"  - {m.model_name}")

Connected to Tinker! Available models:
  - Qwen/Qwen3-235B-A22B-Instruct-2507
  - Qwen/Qwen3-30B-A3B
  - Qwen/Qwen3-30B-A3B-Base
  - Qwen/Qwen3-30B-A3B-Instruct-2507
  - Qwen/Qwen3-32B
  - Qwen/Qwen3-4B-Instruct-2507
  - Qwen/Qwen3-8B
  - Qwen/Qwen3-8B-Base
  - Qwen/Qwen3-VL-235B-A22B-Instruct
  - Qwen/Qwen3-VL-30B-A3B-Instruct
  - Qwen/Qwen3.5-27B
  - Qwen/Qwen3.5-35B-A3B
  - Qwen/Qwen3.5-397B-A17B
  - Qwen/Qwen3.5-397B-A17B:peft:262144
  - Qwen/Qwen3.5-4B


## 2. Load and Explore the Dataset

We load the `Alan123/homo-lumo-grpo` dataset from HuggingFace. This dataset contains chemistry problems related to HOMO-LUMO gap predictions.

> **Note:** Adjust the column names below (`"prompt"`, `"answer"`, etc.) to match the actual schema of your dataset. Run the exploration cell first to see the column names.

In [14]:
#配置代理环境

import os
import socket

# 1. 设置你探测到的 IP 或 OrbStack 专用域名
# 如果 ip route 返回的是 192.168.1.5，就填 http://192.168.1.5:7897
proxy_host = "172.18.48.1" # 如果是纯原生 Linux
# proxy_host = "host.orb.internal" # 如果是 macOS OrbStack
# proxy_host = "172.xx.xx.xx" # 如果是 WSL2/Docker，填 ip route 出来的网关

proxy_url = f"http://{proxy_host}:7897"

# 2. 先做最简单的“端口探活”，不用发 HTTP 请求
def test_port(ip, port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(2)
        return s.connect_ex((ip, port)) == 0

if test_port(proxy_host, 7897):
    print(f"✅ 成功发现代理服务：{proxy_url}")
    os.environ["http_proxy"] = proxy_url
    os.environ["https_proxy"] = proxy_url
    
    # 尝试加载数据
    from datasets import load_dataset
    try:
        # 这里建议先不传 token 试试，如果数据集是公开的
        dataset = load_dataset("Alan123/qa_dataset_with_qed_logp_mw_grpo")
        print("🚀 数据集加载成功！")
    except Exception as e:
        print(f"❌ 代理通了但下载失败: {e}")
else:
    print(f"❌ 依然无法连接到 {proxy_host}:7897。请确认 Clash 的 'Allow LAN' 已开启。")

✅ 成功发现代理服务：http://172.18.48.1:7897
🚀 数据集加载成功！


In [10]:
import tinker
from tinker import types
from tinker_cookbook import renderers, tokenizer_utils

# 1. 配置代理环境 (匹配你 Clash Verge Rev 的 7897 端口)
proxy_host = "172.18.48.1" # 请根据你实际的 WSL/网络 IP 修改
proxy_url = f"http://{proxy_host}:7897"
os.environ["http_proxy"] = proxy_url
os.environ["https_proxy"] = proxy_url

# 2. 从 HuggingFace 加载数据集并截取前 5000 条
print("🚀 正在从 HuggingFace 拉取原数据集...")
dataset = load_dataset("Alan123/qa_dataset_with_qed_logp_mw_grpo", split="train")
subset = dataset.select(range(10))
print(f"✅ 成功截取 {len(subset)} 条样本！")

🚀 正在从 HuggingFace 拉取原数据集...
✅ 成功截取 10 条样本！


In [14]:
import os
import json
import time
from tqdm.auto import tqdm
from datasets import load_dataset, Dataset

# 3. 初始化 Tinker Teacher 客户端
# 建议使用能力更强的模型作为 Teacher，比如 32B 或 30B-A3B
TEACHER_MODEL = "Qwen/Qwen3-30B-A3B" 
LORA_RANK = 8
service_client = tinker.ServiceClient()
teacher_training_client = service_client.create_lora_training_client(
    base_model=TEACHER_MODEL,
    rank=LORA_RANK,
)
# 注：如果你尚未为 Teacher 模型创建 endpoint，可能需要用 service_client.create_sampling_client() 
# 以下假设你可以直接获取 Teacher 模型的采样器
try:
    # teacher_client = service_client.get_sampling_client(TEACHER_MODEL)
    # save_result = teacher_training_client.save_state(name="teacher-init").result()
    teacher_client = teacher_training_client.save_weights_and_get_sampling_client(name="teacher-init")
    
except AttributeError:
    # 兼容 Tinker 不同的 API 命名习惯
    teacher_training_client = service_client.create_lora_training_client(
    base_model=TEACHER_MODEL,
    rank=LORA_RANK,
)
    teacher_client = teacher_training_client.create_sampling_client(TEACHER_MODEL)

tokenizer = tokenizer_utils.get_tokenizer(TEACHER_MODEL)
renderer = renderers.get_renderer('qwen3', tokenizer)

# 4. 定义你的 System Prompt
SYSTEM_PROMPT = """You are a molecular optimization expert.
When asked to modify a molecule, follow these four steps. Output them directly without any preamble or internal thinking block.

[ANALYZE] State the current molecule's QED, LogP, and MW values. Identify which properties need to change and in which direction (higher or lower).

[PLAN] Describe which substructure or functional group you will modify, and explain why this change moves the properties in the required direction.

[EDIT] Output exactly one sentence:
  Replace <old_fragment> with <new_fragment> to form <SMILES>new_smiles_here</SMILES>.

[VERIFY] State the new estimated QED, LogP, and MW. Confirm each property moved in the correct direction.

Example output format:
[ANALYZE] Current molecule: QED=0.65, LogP=4.2, MW=380. Target: QED higher, LogP lower, MW lower.
[PLAN] Remove the tert-butyl group to reduce MW and LogP. Bulky alkyl groups increase LogP significantly.
[EDIT] Replace C(C)(C)C with CH3 to form <SMILES>CC(=O)Nc1ccc(F)cc1</SMILES>.
[VERIFY] New QED≈0.71 (higher ✓), LogP≈2.1 (lower ✓), MW≈293 (lower ✓). All targets met.

Do not output <think> blocks. Start your response directly with [ANALYZE]."""

# 5. 开始批量生成
synthesized_data = []
stop_seqs = renderer.get_stop_sequences()

print(f"🧠 开始调用 {TEACHER_MODEL} 生成四步推理链...")
for i, example in enumerate(tqdm(subset, desc="Generating Data")):
    input_text = example["input"]
    output_text = example["output"]
    
    # 【核心技巧】: 把原始的 target (EDIT 操作) 一并交待给 Teacher，要求它补全逻辑
    #  这里有一个非常大的问题，analyse和verify的部分怎么由LLM直接生成了，应该是我先计算好，然后analyse和verify当中去
    user_content = (
        f"Task:\n{input_text}\n\n"
        f"Expert's Final Edit:\n{output_text}\n\n"
        f"Please reverse-engineer the reasoning and generate the complete 4-step process "
        f"that seamlessly incorporates this Expert's Final Edit into the [EDIT] step."
    )
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content}
    ]
    
    prompt_input = renderer.build_generation_prompt(messages)
    
    try:
        result = teacher_client.sample(
            prompt=prompt_input,
            sampling_params=types.SamplingParams(
                max_tokens=512,
                temperature=0.3,  # 降低温度，确保输出格式的严谨性
                stop=stop_seqs,
            ),
            num_samples=1,
        ).result()
        
        response_text = tokenizer.decode(
            result.sequences[0].tokens,
            skip_special_tokens=True,
        ).replace("<|im_end|>", "").strip()
        
        synthesized_data.append({
            "input": input_text,
            "four_step_output": response_text,  # 这是供我们 SFT 训练的新标签
            "original_output": output_text
        })
        print(synthesized_data[i])
    except Exception as e:
        print(f"⚠️ 第 {i} 条样本生成失败，已跳过。报错: {e}")
        continue
        
    # 定期备份，防止网络中断导致白跑
    if (i + 1) % 500 == 0:
        with open(f"sft_checkpoint_{i+1}.json", "w", encoding="utf-8") as f:
            json.dump(synthesized_data, f, ensure_ascii=False, indent=2)

# 6. 保存最终的 Dataset 到本地
final_dataset = Dataset.from_list(synthesized_data)
final_dataset.save_to_disk("./sft_four_step_dataset_5k")
print("🎉 数据集生成完毕！已保存至当前目录下的 ./sft_four_step_dataset_5k 文件夹。")

APIStatusError: Error code: 402 - {'detail': 'Access for bang.liu@umontreal.ca_default_org is blocked due to billing status. Please add payment at https://tinker-console.thinkingmachines.ai/billing/balance.'}

Session heartbeat failed for 120.0645686100006 seconds for session a581d998-1341-5296-912d-f6ecc65fd8b9. Last exception: APIConnectionError: Connection error..
Your connection may be unreliable or Tinker is down. If this persists, the session will be terminated.
Session heartbeat failed for 135.07334464900032 seconds for session a581d998-1341-5296-912d-f6ecc65fd8b9. Last exception: APIConnectionError: Connection error..
Your connection may be unreliable or Tinker is down. If this persists, the session will be terminated.
Session heartbeat failed for 120.156980578 seconds for session a581d998-1341-5296-912d-f6ecc65fd8b9. Last exception: APIConnectionError: Connection error..
Your connection may be unreliable or Tinker is down. If this persists, the session will be terminated.
Session heartbeat failed for 135.19573044199933 seconds for session a581d998-1341-5296-912d-f6ecc65fd8b9. Last exception: APIConnectionError: Connection error..
Your connection may be unreliable or Tinker is down. 

In [16]:
# ============================================================================
# IMPORTANT: Update these column names based on your dataset's actual schema!
# ============================================================================
# After running the cell above, check the column names and update accordingly.
#
# Common patterns for GRPO chemistry datasets:
#   - "prompt" or "question" or "smiles" or "input" -> the question/molecule
#   - "answer" or "target" or "homo_lumo_gap" or "label" -> the ground truth
#   - "messages" -> pre-formatted chat messages
#
# We provide TWO paths below:
#   Path A: Dataset has "prompt" and "answer" columns
#   Path B: Dataset has pre-formatted "messages" column
#
# Uncomment the path that matches your dataset.

# --- Path A: prompt + answer columns ---
PROMPT_COL = "input"       # <-- UPDATE to your prompt column name
ANSWER_COL = "output"       # <-- UPDATE to your answer column name
DATA_SPLIT = "train"        # <-- UPDATE if needed

# --- Path B: pre-formatted messages ---
# MESSAGES_COL = "messages"  # <-- Uncomment if your dataset uses messages format

raw_data = dataset[DATA_SPLIT]
# raw_data = raw_data_origin[:10000]
print(raw_data)
print(f"Loaded {len(raw_data)} training examples")

Dataset({
    features: ['input', 'output'],
    num_rows: 163415
})
Loaded 163415 training examples


## 3. Configure the Model and Renderer

We'll use **Qwen3-8B** (a good balance of cost and capability). For larger-scale experiments, you can switch to `Qwen/Qwen3-30B-A3B` (MoE, very cost-effective) or `Qwen/Qwen3-32B` (dense).

Tinker uses a **Renderer** to convert chat messages into token sequences with proper special tokens and loss weights.

In [21]:
# Choose your base model
# Options: "Qwen/Qwen3-8B", "Qwen/Qwen3-30B-A3B", "Qwen/Qwen3-32B", etc.
BASE_MODEL = "Qwen/Qwen3-8B"

# LoRA rank: higher = more capacity but costs more. 32 is a good default.
LORA_RANK = 32

print(f"Base model: {BASE_MODEL}")
print(f"LoRA rank: {LORA_RANK}")

Base model: Qwen/Qwen3-8B
LoRA rank: 32


In [22]:
# Set up the renderer for Qwen3 chat format
# The renderer handles special tokens (<|im_start|>, <|im_end|>, etc.)
# and produces proper loss masks for training.

from tinker_cookbook import renderers, tokenizer_utils

tokenizer = tokenizer_utils.get_tokenizer(BASE_MODEL)
renderer = renderers.get_renderer('qwen3', tokenizer)

print(f"Tokenizer vocab size: {tokenizer.vocab_size}")
print(f"Stop sequences: {renderer.get_stop_sequences()}")

Tokenizer vocab size: 151643
Stop sequences: [151645]


## 4. Data Preparation Helpers

We need functions to convert our dataset into the format Tinker expects:
- **For SFT:** `types.Datum` with `model_input` and `loss_fn_inputs` (weights and target_tokens)
- **For GRPO:** Chat messages that can be rendered into prompts for sampling

In [ ]:
# System prompt for chemistry HOMO-LUMO tasks
SYSTEM_PROMPT = (
    """You are a molecular optimization expert.
When asked to modify a molecule, follow these four steps. Output them directly without any preamble or internal thinking block.

[ANALYZE] State the current molecule's QED, LogP, and MW values. Identify which properties need to change and in which direction (higher or lower).

[PLAN] Describe which substructure or functional group you will modify, and explain why this change moves the properties in the required direction.

[EDIT] Output exactly one sentence:
  Replace <old_fragment> with <new_fragment> to form <SMILES>new_smiles_here</SMILES>.

[VERIFY] State the new estimated QED, LogP, and MW. Confirm each property moved in the correct direction.

Example output format:
[ANALYZE] Current molecule: QED=0.65, LogP=4.2, MW=380. Target: QED higher, LogP lower, MW lower.
[PLAN] Remove the tert-butyl group to reduce MW and LogP. Bulky alkyl groups increase LogP significantly.
[EDIT] Replace C(C)(C)C with CH3 to form <SMILES>CC(=O)Nc1ccc(F)cc1</SMILES>.
[VERIFY] New QED≈0.71 (higher ✓), LogP≈2.1 (lower ✓), MW≈293 (lower ✓). All targets met.

Do not output <think> blocks. Start your response directly with [ANALYZE]."""
)



def make_messages(example):
    """
    Convert a raw dataset example into a list of chat messages.
    Update this function based on your dataset's column schema.
    """
    # --- Path A: prompt + answer columns ---
    prompt_text = example[PROMPT_COL]
    answer_text = str(example[ANSWER_COL])

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt_text},
        {"role": "assistant", "content": f"{answer_text}"}
    ]
    return messages

    # --- Path B: pre-formatted messages ---
    # return example[MESSAGES_COL]


def make_prompt_messages(example, system_prompt = SYSTEM_PROMPT):
    """
    Create the prompt-only messages (no assistant response) for RL sampling.
    """
    prompt_text = example[PROMPT_COL]
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt_text},
    ]
    return messages

In [ ]:
def prepare_sft_datum(example):
    messages = make_messages(example)
    model_input, weights = renderer.build_supervised_example(messages)

    input_tokens = model_input.to_ints()
    weights_list = weights.tolist() if hasattr(weights, 'tolist') else list(weights)

    target_tokens = input_tokens[1:] + [input_tokens[-1]]
    weights_shifted = weights_list[1:] + [0]

    return types.Datum(
        model_input=types.ModelInput.from_ints(tokens=input_tokens[:-1]),
        loss_fn_inputs=dict(
            weights=weights_shifted[:-1],
            target_tokens=target_tokens[:-1],
        ),
    )



# Test: prepare one datum and inspect
test_datum = prepare_sft_datum(raw_data[0])
n_tokens = len(test_datum.model_input.to_ints())

# Debug: check what type the weights actually are
weights_val = test_datum.loss_fn_inputs['weights']
print(f"Weights type: {type(weights_val)}")
if hasattr(weights_val, '__len__') and len(weights_val) > 0:
    print(f"First element type: {type(weights_val[0])}, value: {weights_val[0]}")

## 5. Stage 1: Supervised Fine-Tuning (SFT)

SFT teaches the model the basic format and chemistry knowledge. We use the `cross_entropy` loss function, which trains the model to predict the next token for the assistant's response.

**Tinker's training loop:**
1. Prepare a batch of `Datum` objects
2. Call `forward_backward()` to compute gradients
3. Call `optim_step()` to update the LoRA weights
4. Repeat

In [ ]:
# ========================
# SFT Hyperparameters
# ========================
SFT_LEARNING_RATE = 1e-4
SFT_BATCH_SIZE = 4           # Number of examples per gradient step
SFT_NUM_STEPS =  50          # Total training steps (adjust based on dataset size)
SFT_EVAL_EVERY = 10          # Evaluate every N steps
SFT_GRAD_ACCUM_STEPS = 1     # Gradient accumulation steps

print(f"SFT config: lr={SFT_LEARNING_RATE}, batch={SFT_BATCH_SIZE}, steps={SFT_NUM_STEPS}")

In [ ]:
# Create the training client for SFT
sft_training_client = service_client.create_lora_training_client(
    base_model=BASE_MODEL,
    rank=LORA_RANK,
)
sft_training_client.load_state_with_optimizer("tinker://88c08ced-d3ee-5776-b214-1e7ef8063c76:train:0/weights/MultistepRL-sft-final.weekend.Qwen8B")
print(f"SFT Training client created (run ID visible in Tinker console)")

In [ ]:
# Prepare all SFT data
print("Preparing SFT training data...")
sft_data = []
for i, example in enumerate(raw_data):
    try:
        datum = prepare_sft_datum(example)
        sft_data.append(datum)
    except Exception as e:
        if i < 3:
            print(f"  Warning: skipping example {i}: {e}")

print(f"Prepared {len(sft_data)} SFT examples")

# Split into train/test
split_idx = int(0.9 * len(sft_data))
sft_train = sft_data[:split_idx]
sft_test = sft_data[split_idx:]
print(f"Train: {len(sft_train)}, Test: {len(sft_test)}")

In [ ]:
# ========================
# SFT Training Loop
# ========================
sft_metrics = []
rng = np.random.default_rng(42)

print("Starting SFT training...")
print(f"{'Step':>5} | {'Train Loss':>10} | {'Test Loss':>10} | {'Time (s)':>8}")
print("-" * 50)

for step in range(SFT_NUM_STEPS):
    t0 = time.time()

    # Sample a mini-batch
    batch_indices = rng.choice(len(sft_train), size=SFT_BATCH_SIZE, replace=False)
    batch = [sft_train[i] for i in batch_indices]

    # Forward-backward pass with cross-entropy loss
    fwdbwd_future = sft_training_client.forward_backward(batch, loss_fn="cross_entropy")

    # Optimizer step
    optim_future = sft_training_client.optim_step(
        types.AdamParams(learning_rate=SFT_LEARNING_RATE)
    )

    # Wait for results
    fwdbwd_result = fwdbwd_future.result()
    optim_result = optim_future.result()

    # Compute train loss
    logprobs = np.concatenate(
        [out['logprobs'].tolist() for out in fwdbwd_result.loss_fn_outputs]
    )
    weights = np.concatenate(
        [d.loss_fn_inputs['weights'].tolist() if hasattr(d.loss_fn_inputs['weights'], 'tolist')
         else d.loss_fn_inputs['weights']
         for d in batch]
    )
    weights_arr = np.array(weights, dtype=float)
    train_loss = -np.dot(logprobs, weights_arr) / max(weights_arr.sum(), 1)

    elapsed = time.time() - t0
    metrics_entry = {"step": step, "train_loss": float(train_loss), "time": elapsed}

    # Periodic evaluation on test set
    if step % SFT_EVAL_EVERY == 0 and len(sft_test) > 0:
        test_batch = sft_test[:min(SFT_BATCH_SIZE, len(sft_test))]
        test_fwdbwd = sft_training_client.forward_backward(test_batch, loss_fn="cross_entropy")
        # Note: we don't call optim_step for test — just forward pass
        test_result = test_fwdbwd.result()
        test_logprobs = np.concatenate(
            [out['logprobs'].tolist() for out in test_result.loss_fn_outputs]
        )
        test_weights = np.concatenate(
            [d.loss_fn_inputs['weights'].tolist() if hasattr(d.loss_fn_inputs['weights'], 'tolist')
             else d.loss_fn_inputs['weights']
             for d in test_batch]
        )
        test_weights_arr = np.array(test_weights, dtype=float)
        test_loss = -np.dot(test_logprobs, test_weights_arr) / max(test_weights_arr.sum(), 1)
        metrics_entry["test_loss"] = float(test_loss)
        print(f"{step:>5} | {train_loss:>10.4f} | {test_loss:>10.4f} | {elapsed:>8.1f}")
    else:
        print(f"{step:>5} | {train_loss:>10.4f} | {'—':>10} | {elapsed:>8.1f}")

    sft_metrics.append(metrics_entry)

print("\nSFT training complete!")

In [ ]:
# Save the SFT checkpoint
# This saves the LoRA weights + optimizer state so we can:
#   1. Sample from the SFT model
#   2. Continue training with GRPO

sft_save_result = sft_training_client.save_state(name="MultistepRL-sft-final.weekend.Qwen8B").result()
sft_checkpoint_path = sft_save_result.path
print(f"SFT checkpoint saved at: {sft_checkpoint_path}")

# Also save a sampler-only checkpoint (lighter weight)
sft_sampling_client = sft_training_client.save_weights_and_get_sampling_client(
    name="sft-sampler.weekend.Qwen8B"
)
print("SFT sampling client ready")

In [ ]:
# Test the SFT model with a sample prediction
test_example = raw_data[0]
test_prompt_msgs = make_prompt_messages(test_example)
prompt_input = renderer.build_generation_prompt(test_prompt_msgs)
stop_seqs = renderer.get_stop_sequences()

sample_result = sft_sampling_client.sample(
    prompt=prompt_input,
    sampling_params=types.SamplingParams(
        max_tokens=512,
        temperature=0.7,
        stop=stop_seqs,
    ),
    num_samples=1,
).result()

response_text = tokenizer.decode(
    sample_result.sequences[0].tokens,
    skip_special_tokens=True,
).replace("<|im_end|>", "").strip()
print("=" * 60)
print("PROMPT:", test_example[PROMPT_COL][:500])
print("=" * 60)
print("MODEL RESPONSE (SFT):")
print(response_text)
print("=" * 60)
print("GROUND TRUTH:", test_example[ANSWER_COL])

## 6. Plot SFT Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot([m['step'] for m in sft_metrics], [m['train_loss'] for m in sft_metrics],
        label='Train Loss', alpha=0.7)
test_metrics = [m for m in sft_metrics if 'test_loss' in m]
if test_metrics:
    ax.plot([m['step'] for m in test_metrics], [m['test_loss'] for m in test_metrics],
            'o-', label='Test Loss')
ax.set_xlabel('Step')
ax.set_ylabel('Loss (NLL per token)')
ax.set_title('SFT Training Curve')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 7. Stage 2: Reinforcement Learning

Now we switch to **GRPO-style RL training** to improve the model's accuracy using a reward signal.

**How GRPO works with Tinker:**
1. For each prompt, **sample G completions** from the current model
2. **Score** each completion with a reward function
3. Compute **group-normalized advantages** (subtract mean, divide by std within each group)
4. Use the `ppo` (or `importance_sampling`) loss to update the policy

Tinker's built-in `ppo` loss handles the clipped surrogate objective. The GRPO-specific part (group normalization of advantages) is done on the client side.

---

## 关于qed logp mw的rewardFunc

In [3]:
import re
from rdkit import Chem
from rdkit import DataStructs
from rdkit.Chem import QED
from rdkit.Chem import Descriptors
from rdkit.Chem import Crippen
from rdkit.Chem import rdFingerprintGenerator
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")
from typing import Dict, List, Tuple
from collections import defaultdict


def _canonical_smiles(smiles_text: str) -> str | None:
    """Return canonical SMILES, or None if parsing fails."""
    if not smiles_text:
        return None
    mol = Chem.MolFromSmiles(smiles_text)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol)


def strict_format_reward(completions: List[str], **kwargs) -> List[float]:
    """
    +1.0 if the completion matches:
      'Replace ... with ... to form <SMILES>...</SMILES>.'   OR
      'Add ... to form <SMILES>...</SMILES>.'                OR
      'Remove ... to form <SMILES>...</SMILES>.'
    else -4.0
    """
    pat = re.compile(
        r"^(Replace|Add|Remove)\s+(.+?)(?:\s+with\s+(.+?))?\s+to\s+form\s+<SMILES>(.+)</SMILES>\.$",
        re.DOTALL,
    )
    rewards = []
    for text in completions:
        m = pat.fullmatch(text.strip())
        rewards.append(1.0 if m else -4.0)
    return rewards


_completion_counter = defaultdict(int)
_fragment_counter = defaultdict(int)


def repetition_penalty_reward(completions: List[str], **kwargs) -> List[float]:
    """
    Penalize repeated final SMILES and overused replacement fragments.
    """
    rewards = []
    for text in completions:
        penalty = 0.0

        # SMILES repetition / invalidity
        m = re.search(r"<SMILES>(.*?)</SMILES>", text)
        if m:
            smi = m.group(1)
            can = _canonical_smiles(smi)
            if can:
                count = _completion_counter[can]
                _completion_counter[can] += 1
                penalty += -0.1 * count
            else:
                penalty += -1.0

        # Replacement fragment repetition
        m2 = re.match(
            r"^(Replace|Add)\s+(.+?)(?:\s+with\s+(.+?))?\s+to\s+form\s+<SMILES>",
            text.strip(),
        )
        if m2:
            repl = (m2.group(3) or "").strip()
            if repl:
                can_frag = _canonical_smiles(repl)
                if can_frag:
                    count_f = _fragment_counter[can_frag]
                    _fragment_counter[can_frag] += 1
                    penalty += -0.05 * count_f
                else:
                    penalty += -0.5

        rewards.append(penalty)
    return rewards


def multi_property_reward(completions: List[str], **kwargs) -> List[float]:
    """
    Reward in (-inf, 1], 1 - (weighted, direction-aware normalized error),
    using targets parsed from the prompt:
      <QED>x</QED> (higher|lower), <LogP>y</LogP> (higher|lower), <MW>z</MW> (higher|lower)
    """
    prompts: List[str] = kwargs["prompts"]
    out = []

    for comp, prompt in zip(completions, prompts):
        in_m = re.search(r"<SMILES>(.*?)</SMILES>", prompt)
        out_m = re.search(r"<SMILES>(.*?)</SMILES>", comp)
        qed_m = re.search(r"<QED>(-?\d+\.\d+)</QED>\s*(higher|lower)", prompt)
        logp_m = re.search(r"<LogP>(-?\d+\.\d+)</LogP>\s*(higher|lower)", prompt)
        mw_m = re.search(r"<MW>(-?\d+\.\d+)</MW>\s*(higher|lower)", prompt)

        if not (in_m and out_m and qed_m and logp_m and mw_m):
            out.append(-4.0)
            continue

        s_in, s_out = in_m.group(1), out_m.group(1)
        m_in, m_out = Chem.MolFromSmiles(s_in), Chem.MolFromSmiles(s_out)
        if not (m_in and m_out):
            out.append(-4.0)
            continue

        # Originals
        qed_in = QED.qed(m_in)
        logp_in = Crippen.MolLogP(m_in)
        mw_in = Descriptors.ExactMolWt(m_in)

        # Generated
        qed_out = QED.qed(m_out)
        logp_out = Crippen.MolLogP(m_out)
        mw_out = Descriptors.ExactMolWt(m_out)

        # Parse targets from deltas + direction
        def _parse(m, base):
            delta = float(m.group(1))
            direction = 1 if m.group(2) == "higher" else -1
            return base + direction * delta, direction

        qed_t, d_qed = _parse(qed_m, qed_in)
        logp_t, d_logp = _parse(logp_m, logp_in)
        mw_t, d_mw = _parse(mw_m, mw_in)

        # Clamp QED target to [0.01, 0.99]
        qed_t = max(0.01, min(0.99, qed_t))

        # Normalized absolute errors
        qed_err = abs(qed_out - qed_t) / 0.5
        logp_err = abs(logp_out - logp_t) / 3.0
        mw_err = abs(mw_out - mw_t) / 50.0

        # Wrong-direction penalties
        dir_pen_qed = 2 if (qed_out - qed_in) * d_qed < 0 else 1
        dir_pen_logp = 2 if (logp_out - logp_in) * d_logp < 0 else 1
        dir_pen_mw = 2 if (mw_out - mw_in) * d_mw < 0 else 1

        total_err = (
            qed_err * dir_pen_qed
            + logp_err * dir_pen_logp
            + mw_err * dir_pen_mw
        )
        reward = 1.0 - total_err
        out.append(reward)
    return out


REWARD_WEIGHTS = {
    "strict": 0.25,
    "repetition": 0.15,
    "property": 0.60,
}


def compute_total_reward(completions: List[str], prompts: List[str]) -> Tuple[List[float], Dict[str, List[float]]]:
    """Combine the new reward stack only: strict + repetition + property."""
    if len(completions) != len(prompts):
        raise ValueError("completions and prompts must have the same length")

    strict_scores = strict_format_reward(completions)
    repetition_scores = repetition_penalty_reward(completions)
    property_scores = multi_property_reward(completions, prompts=prompts)

    totals = []
    for s, r, p in zip(strict_scores, repetition_scores, property_scores):
        totals.append(
            REWARD_WEIGHTS["strict"] * s
            + REWARD_WEIGHTS["repetition"] * r
            + REWARD_WEIGHTS["property"] * p
        )

    details = {
        "strict": strict_scores,
        "repetition": repetition_scores,
        "property": property_scores,
    }
    return totals, details

In [4]:
"""
======================
面向分子属性优化任务的多步强化学习 Reward 系统
基于 Tinker API + GRPO，支持两种训练模式：

  模式 A（单轮 PRM）：模型一次性输出结构化推理链，
                     事后对每个步骤打分，奖励回溯分配。

  模式 B（多轮 RL） ：模型逐步输出，RDKit 环境在每步后
                     真实运行并注入反馈，奖励通过折扣回报积累。

两种模式共享同一套 Step-level Reward 函数（score_step）。
唯一的结构差异在于 _run_env() 是否在步骤之间被调用。

"""



RDLogger.DisableLog("rdApp.*")


# ══════════════════════════════════════════════════════════════════════════════
#  第一部分：Step-level Reward 函数（两种模式共用）
#
#  设计思路：
#    每个步骤有独立的、局部可验证的目标，不依赖最终分子是否命中属性。
#    这样即使模型最终失败，中间步骤做对了也能得到正向梯度信号。
#
#    步骤        局部目标                    信号来源
#    ──────────  ──────────────────────────  ────────────────────
#    ANALYZE     正确识别属性名和变化方向      prompt 标签匹配
#    PLAN        包含合理的化学编辑意图        化学关键词覆盖
#    EDIT        输出合法 SMILES + 方向正确   RDKit 实时计算
#    VERIFY      数值自洽，声称与实际一致      数字 + 正向词检测
# ══════════════════════════════════════════════════════════════════════════════

def parse_reasoning_steps(completion_text: str) -> dict[str, str]:
    """
    从模型 completion 中解析出四个结构化步骤。

    期望模型输出格式（由 system prompt 约束，见 SYSTEM_PROMPT_MULTISTEP）：
        [ANALYZE] 当前分子 QED=0.72, LogP=3.1 ...
        [PLAN]    需要降低 LogP，考虑去除卤素 ...
        [EDIT]    Replace Cl with F to form <SMILES>CCO</SMILES>
        [VERIFY]  新 QED=0.74，方向正确

    若某步骤缺失，对应键值为空字符串，不影响其他步骤打分。
    """
    # 先清理残留的 thinking token（skip_special_tokens 有时不完整）
    text = re.sub(r"<think>.*?</think>", "", completion_text, flags=re.DOTALL).strip()

    steps = {}
    for tag in ["ANALYZE", "PLAN", "EDIT", "VERIFY"]:
        m = re.search(rf"\[{tag}\](.*?)(?=\[(?:ANALYZE|PLAN|EDIT|VERIFY)\]|\Z)", text, re.DOTALL)
        steps[tag] = m.group(1).strip() if m else ""

    # Fallback：如果四步都是空的，说明模型忽略了标签格式
    # 尝试从 thinking block 里提取（最后的救援）
    if all(v == "" for v in steps.values()):
        think_m = re.search(r"<think>(.*?)</think>", completion_text, re.DOTALL)
        if think_m:
            think_text = think_m.group(1)
            # 把 thinking 内容当作 ANALYZE+PLAN 的合并体
            steps["ANALYZE"] = think_text[:500]  # 截断避免过长
            steps["PLAN"]    = think_text[500:] if len(think_text) > 500 else ""

        # EDIT 从全文找 <SMILES>（无论在哪个位置）
        edit_m = re.search(
            r"(Replace|Add|Remove)\s+.+?\s+to\s+form\s+<SMILES>.+?</SMILES>",
            completion_text, re.DOTALL
        )
        if edit_m:
            steps["EDIT"] = edit_m.group(0)

    return steps


def r_analyze(step_text: str, prompt_text: str) -> float:
    """
    评分范围：[0.0, 0.5]

    逻辑：对 QED、LogP、MW 三个属性分别检查两件事：
      1. 模型是否提及该属性名（体现感知能力）
      2. 模型是否正确识别了变化方向 higher/lower（体现理解能力）
    每项各奖励 0.1 分，上限 0.5。
    """
    if not step_text:
        return 0.0
    score = 0.0
    for prop in ["QED", "LogP", "MW"]:
        if re.search(prop, step_text, re.IGNORECASE):
            score += 0.1
        pm = re.search(
            rf"<{prop}>([\d.]+)</{prop}>\s*(higher|lower)", prompt_text
        )
        if pm and pm.group(2) in step_text.lower():
            score += 0.1
    return min(score, 0.5)


def r_plan(step_text: str, prompt_text: str) -> float:
    """
    评分范围：[0.0, 0.4]

    逻辑：检查 PLAN 步骤是否包含实质性的化学编辑意图。
    使用关键词列表做轻量打分，无需 RDKit，计算极快。
    每命中一个关键词得 0.08 分，上限 0.4。

    注：此处 prompt_text 保留为参数供未来扩展（例如基于输入分子
    的子结构匹配来判断 plan 的化学合理性）。
    """
    chemical_keywords = [
        "replace", "add", "remove", "substitute",
        "ring", "group", "bond", "atom",
        "hydroxyl", "amine", "carbonyl", "halogen",
        "aromatic", "aliphatic", "chain", "branch",
    ]
    hits = sum(1 for kw in chemical_keywords if kw in step_text.lower())
    return min(hits * 0.08, 0.4)


def r_edit(step_text: str, prompt_text: str) -> float:
    """
    评分范围：[-1.0, 1.0]，这是最重要的 process reward。

    评分层次（由低到高）：
      -1.0  没有 <SMILES> 标签（格式完全错误）
      -0.5  SMILES 标签存在但分子非法
       0.1  SMILES 合法，但无法提取输入分子做对比
       0.3  SMILES 合法（基础分）
       0.3 + 0.5 × direction_score  方向正确时获得额外奖励，最高 1.0

    方向评分（_direction_reward）：对三个属性分别检查实际变化
    方向是否与 prompt 要求一致，取平均值。
    """
    smiles_m = re.search(r"<SMILES>(.*?)</SMILES>", step_text)
    if not smiles_m:
        return -1.0

    mol_out = Chem.MolFromSmiles(smiles_m.group(1))
    if mol_out is None:
        return -0.5

    smi_in_m = re.search(r"<SMILES>(.*?)</SMILES>", prompt_text)
    if not smi_in_m:
        return 0.1

    mol_in = Chem.MolFromSmiles(smi_in_m.group(1))
    if mol_in is None:
        return 0.1

    direction_score = _direction_reward(mol_in, mol_out, prompt_text)
    return min(0.3 + direction_score * 0.5, 1.0)  # todo： 这里逻辑有问题


def r_verify(step_text: str, prompt_text: str) -> float:
    """
    评分范围：[0.0, 0.3]

    逻辑：检查 VERIFY 步骤是否体现了自我一致性。
    两个软指标：数字出现（说明模型真的计算了属性）+
    正向词出现（说明模型认为方向正确）。
    两者都有得 0.3，有其一得 0.15，都没有得 0。

    注：此处不做"声称值 vs 实际值"的精确核对——
    那需要再次调用 RDKit，成本较高，留给 outcome reward 覆盖。
    """
    has_numbers = bool(re.search(r"\d+\.\d+", step_text))
    positive_words = ["correct", "improved", "higher", "lower", "better", "achieved"]
    has_positive = any(w in step_text.lower() for w in positive_words)

    if has_numbers and has_positive:
        return 0.3
    elif has_numbers or has_positive:
        return 0.15
    return 0.0


def _direction_reward(mol_in, mol_out, prompt_text: str) -> float:
    """
    辅助函数：计算三个属性方向匹配的平均得分。

    对每个属性：实际变化方向与 prompt 要求一致得 1.0，否则得 0.0。
    只对 prompt 中明确标注了 higher/lower 的属性计分，取平均。
    """
    prop_fns = {
        "QED":  (QED.qed,              QED.qed(mol_in),              QED.qed(mol_out)),
        "LogP": (Crippen.MolLogP,      Crippen.MolLogP(mol_in),      Crippen.MolLogP(mol_out)),
        "MW":   (Descriptors.ExactMolWt, Descriptors.ExactMolWt(mol_in), Descriptors.ExactMolWt(mol_out)),
    }
    scores = []
    for prop, (_, v_in, v_out) in prop_fns.items():
        m = re.search(rf"<{prop}>([\d.]+)</{prop}>\s*(higher|lower)", prompt_text)
        if not m:
            continue
        required = 1 if m.group(2) == "higher" else -1
        actual   = 1 if (v_out - v_in) > 0 else -1
        scores.append(1.0 if required == actual else 0.0)
    return sum(scores) / len(scores) if scores else 0.0


def score_step(tag: str, text: str, context: dict) -> float:
    """
    统一入口：根据步骤标签分发到对应的 reward 函数。

    context 需包含：
      - prompt_text (str)：原始 prompt，含 <QED>/<LogP>/<MW> 标签
      - mol_in (Chem.Mol | None)：解析自 prompt 的输入分子对象
    """
    prompt_text = context["prompt_text"]
    if tag == "ANALYZE":
        return r_analyze(text, prompt_text)
    elif tag == "PLAN":
        return r_plan(text, prompt_text)
    elif tag == "EDIT":
        return r_edit(text, prompt_text)
    elif tag == "VERIFY":
        return r_verify(text, prompt_text)
    return 0.0


In [5]:

# ══════════════════════════════════════════════════════════════════════════════
#  第二部分：Reward 权重与聚合
#
#  权重分配思路：
#    - outcome reward 占 50%，保证最终化学目标不被稀释
#    - EDIT 步骤是分子生成的核心行为，权重最高（25%）
#    - ANALYZE 和 PLAN 是推理质量的体现，各 10%
#    - VERIFY 是自洽性检查，权重最低（5%）
#
#  这套权重在两种模式下完全一致，便于对比实验。
# ══════════════════════════════════════════════════════════════════════════════

STEP_WEIGHTS = {
    "ANALYZE": 0.05,
    "PLAN":    0.15,
    "EDIT":    0.25,
    "VERIFY":  0.05,
}
OUTCOME_WEIGHT = 0.50  # 与 step rewards 之和为 1.0


def aggregate_step_rewards(step_rewards: dict[str, float]) -> float:
    """将四个步骤的得分按权重聚合为 process reward 小计（不含 outcome）。"""
    return sum(STEP_WEIGHTS.get(tag, 0.0) * r for tag, r in step_rewards.items())


def compute_group_advantages(rewards: list[float]) -> list[float]:
    """
    GRPO 风格的组内归一化：减去组均值、除以组标准差。
    若组内所有奖励相同（std ≈ 0），返回全零，避免 NaN。
    """
    arr = np.array(rewards, dtype=np.float64)
    std = arr.std()
    if std < 1e-8:
        return [0.0] * len(rewards)
    return ((arr - arr.mean()) / std).tolist()


In [6]:
# ══════════════════════════════════════════════════════════════════════════════
#  第三部分：辅助函数（两种模式共用）
# ══════════════════════════════════════════════════════════════════════════════

import re
import numpy as np
from typing import Optional

import tinker
from tinker import types
from rdkit import Chem, RDLogger
from rdkit.Chem import QED, Descriptors, Crippen

def _mol_from_prompt(prompt_text: str) -> Optional[Chem.Mol]:
    """从 prompt 文本的 <SMILES> 标签中提取输入分子。"""
    m = re.search(r"<SMILES>(.*?)</SMILES>", prompt_text)
    if not m:
        return None
    return Chem.MolFromSmiles(m.group(1))


def _check_targets(mol: Chem.Mol, prompt_text: str) -> list[str]:
    """
    返回尚未达标的属性名列表。
    达标标准：实际变化方向与 prompt 要求一致（不要求精确命中数值）。
    """
    mol_in = _mol_from_prompt(prompt_text)
    if mol_in is None:
        return []

    prop_fns = {
        "QED":  QED.qed,
        "LogP": Crippen.MolLogP,
        "MW":   Descriptors.ExactMolWt,
    }
    misses = []
    for prop, fn in prop_fns.items():
        m = re.search(rf"<{prop}>([\d.]+)</{prop}>\s*(higher|lower)", prompt_text)
        if not m:
            continue
        required = 1 if m.group(2) == "higher" else -1
        if (fn(mol) - fn(mol_in)) * required <= 0:
            misses.append(prop)
    return misses


def _run_env(edit_step_text: str, prompt_text: str) -> tuple[str, Optional[Chem.Mol]]:
    """
    【仅模式 B 使用】调用 RDKit 计算真实属性，生成结构化文字反馈。

    模式 A 不调用此函数——这是两种模式唯一的结构差异：
      模式 A：reward 事后计算，环境不介入生成过程。
      模式 B：每步结束后此函数运行，反馈注入下一轮对话。

    返回值：(反馈文本, 合法分子对象 或 None)
    """
    smi_m = re.search(r"<SMILES>(.*?)</SMILES>", edit_step_text)
    if not smi_m:
        return "Your output did not contain a valid SMILES. Please try again.", None

    mol = Chem.MolFromSmiles(smi_m.group(1))
    if mol is None:
        return "The SMILES you provided is chemically invalid. Please revise.", None

    qed_val  = QED.qed(mol)
    logp_val = Crippen.MolLogP(mol)
    mw_val   = Descriptors.ExactMolWt(mol)

    misses = _check_targets(mol, prompt_text)
    hint = (
        f"Properties still not meeting targets: {', '.join(misses)}. Please refine."
        if misses else
        "All targets met. Please verify your reasoning."
    )

    feedback = (
        f"Environment feedback: "
        f"QED={qed_val:.3f}, LogP={logp_val:.2f}, MW={mw_val:.1f}. {hint}"
    )
    return feedback, mol


In [7]:
# System prompt：要求模型严格输出四步结构（两种模式均使用）
SYSTEM_PROMPT_MULTISTEP = """You are a molecular optimization expert.
When asked to modify a molecule, follow these four steps. Output them directly without any preamble or internal thinking block.

[ANALYZE] State the current molecule's QED, LogP, and MW values. Identify which properties need to change and in which direction (higher or lower).

[PLAN] Describe which substructure or functional group you will modify, and explain why this change moves the properties in the required direction.

[EDIT] Output exactly one sentence:
  Replace <old_fragment> with <new_fragment> to form <SMILES>new_smiles_here</SMILES>.

[VERIFY] State the new estimated QED, LogP, and MW. Confirm each property moved in the correct direction.

Example output format:
[ANALYZE] Current molecule: QED=0.65, LogP=4.2, MW=380. Target: QED higher, LogP lower, MW lower.
[PLAN] Remove the tert-butyl group to reduce MW and LogP. Bulky alkyl groups increase LogP significantly.
[EDIT] Replace C(C)(C)C with CH3 to form <SMILES>CC(=O)Nc1ccc(F)cc1</SMILES>.
[VERIFY] New QED≈0.71 (higher ✓), LogP≈2.1 (lower ✓), MW≈293 (lower ✓). All targets met.

Do not output <think> blocks. Start your response directly with [ANALYZE]."""

In [8]:
# ══════════════════════════════════════════════════════════════════════════════
#  第四部分：模式 A —— 单轮 PRM（Process Reward Model）
#
#  执行流程：
#    1. 用当前策略对同一 prompt 采样 G 个 completion（GRPO 标准做法）
#    2. 解析每个 completion 的四个步骤
#    3. 对每个步骤调用 score_step，加权求和得 process reward
#    4. 叠加来自原有 compute_total_reward 的 outcome reward
#    5. 组内归一化得 advantage，构造 Tinker Datum
#
#  优点：实现简单，与现有 GRPO 循环几乎零改动，适合作为起步阶段。
#  限制：模型无法根据环境反馈在同一 episode 内修正错误。
# ══════════════════════════════════════════════════════════════════════════════

def run_mode_a(
    example: dict,
    renderer,
    sampling_client,
    tokenizer,
    num_samples: int = 4,
    max_tokens: int = 512,
    temperature: float = 0.7,
) -> tuple[list[types.Datum], list[float], dict]:
    """
    单轮 PRM 模式：采样多个 completion，逐步打分，返回训练 Datum 列表。

    返回值：
      datums   - 可直接传入 forward_backward 的 Datum 列表
      rewards  - 每个样本的总奖励（用于日志）
      details  - 各子项奖励的明细字典（用于监控）
    """
    prompt_text = str(example[PROMPT_COL])
    context = {"prompt_text": prompt_text, "mol_in": _mol_from_prompt(prompt_text)}

    # 每个 prompt 组重置重复惩罚计数器，避免跨组污染奖励尺度
    if "_completion_counter" in globals():
        _completion_counter.clear()
    if "_fragment_counter" in globals():
        _fragment_counter.clear()

    prompt_msgs = make_prompt_messages(example, system_prompt=SYSTEM_PROMPT_MULTISTEP)
    prompt_input = renderer.build_generation_prompt(prompt_msgs)
    prompt_tokens = prompt_input.to_ints()

    result = sampling_client.sample(
        prompt=prompt_input,
        sampling_params=types.SamplingParams(
            max_tokens=max_tokens,
            temperature=temperature,
            stop=renderer.get_stop_sequences(),
        ),
        num_samples=num_samples,
    ).result()

    raw_rewards   = []
    detail_log    = []
    comp_data     = []  # 暂存，等组归一化后再构造 Datum

    for seq in result.sequences:
        completion_text = tokenizer.decode(
            seq.tokens,
            skip_special_tokens=True,
        ).replace("<|im_end|>", "").strip()
        steps = parse_reasoning_steps(completion_text)

        step_rewards = {tag: score_step(tag, text, context) for tag, text in steps.items()}
        process_r = aggregate_step_rewards(step_rewards)

        edit_text = steps.get("EDIT", "")
        outcome_scores, _ = compute_total_reward([edit_text], prompts=[prompt_text])
        outcome_r = outcome_scores[0]

        total_r = process_r + OUTCOME_WEIGHT * outcome_r
        raw_rewards.append(total_r)
        detail_log.append({**step_rewards, "process": process_r, "outcome": outcome_r, "total": total_r})

        comp_data.append({
            "comp_tokens": list(seq.tokens),
            "comp_logprobs": list(seq.logprobs),
        })

    # GRPO 组归一化
    advantages = compute_group_advantages(raw_rewards)
    n_p = len(prompt_tokens)

    datums = []
    for cd, adv in zip(comp_data, advantages):
        if adv == 0.0:
            continue
        comp_tokens = cd["comp_tokens"]
        comp_logprobs = cd["comp_logprobs"]
        full_tokens = prompt_tokens + comp_tokens
        n_c = len(comp_tokens)
        min_len = min(len(full_tokens) - 1, n_p - 1 + n_c, n_p - 1 + len(comp_logprobs))

        datums.append(types.Datum(
            model_input=types.ModelInput.from_ints(tokens=full_tokens[:-1]),
            loss_fn_inputs=dict(
                target_tokens=full_tokens[1:][:min_len],
                advantages=([0.0] * (n_p - 1) + [adv] * n_c)[:min_len],
                logprobs=([0.0] * (n_p - 1) + comp_logprobs)[:min_len],
            ),
        ))

    # 聚合 detail 用于日志输出
    details = {
        k: float(np.mean([d.get(k, 0.0) for d in detail_log]))
        for k in ["ANALYZE", "PLAN", "EDIT", "VERIFY", "process", "outcome", "total"]
    }
    return datums, raw_rewards, details




In [9]:
# ══════════════════════════════════════════════════════════════════════════════
#  修复5：Mode B 的组归一化 + EMA baseline
#
#  设计思路：
#    两个机制各司其职，解决不同层面的问题：
#
#    ① 组归一化（intra-prompt）
#       对同一 prompt 采 K 条轨迹，计算每条轨迹的折扣回报 G_k，
#       再做 (G_k - mean) / std 归一化。
#       作用：告诉模型"在同一起点下，哪种推理路径更优"。
#       这是 GRPO 的核心信号，Mode B 必须保留。
#
#    ② EMA baseline（inter-prompt）
#       维护一个跨 prompt、跨 iteration 的滑动平均 baseline。
#       在组归一化之后再减去 baseline（可选，作为二次校正）。
#       作用：防止某些 prompt 天然容易（reward 整体高）导致
#       advantage 尺度膨胀，稳定不同 prompt 之间的梯度大小。
#
#    两者的关系：
#      组归一化是"排名"，EMA 是"定零点"。
#      单独用 EMA（原始建议）：有零点但没有排名 → 信号弱。
#      只用组归一化：排名准确但跨 prompt 尺度不齐 → 梯度方差大。
#      两者结合：既有排名又有稳定尺度 → 最优。
#
#    关于 K 的选择：
#      Mode A：K=4~8（每条只需 1 次采样）
#      Mode B：K=2~3（每条需要 max_turns 次采样，成本是 Mode A 的 max_turns 倍）
#      K=1 时退化为普通 PG，组归一化失效，不建议使用。
# ══════════════════════════════════════════════════════════════════════════════
 
class EMABaseline:
    """
    指数移动平均 baseline，用于跨 prompt 稳定 advantage 的绝对尺度。
 
    update(G) 先返回当前 baseline 再更新，使得当前 batch 使用
    "历史经验"而非"本轮数据"作为参考，避免自举偏差。
    """
    def __init__(self, alpha: float = 0.05, init: float = 0.0):
        self.alpha   = alpha
        self.value   = init
        self._warmed = False  # 前几个样本用来热身，不参与训练
 
    def update_and_get(self, G: float) -> float:
        """返回当前 baseline，然后更新。"""
        current = self.value
        if not self._warmed:
            # 第一个样本直接初始化，避免初始 baseline=0 造成偏差
            self.value   = G
            self._warmed = True
        else:
            self.value = (1 - self.alpha) * self.value + self.alpha * G
        return current

In [10]:
# ══════════════════════════════════════════════════════════════════════════════
#  第五部分：模式 B —— 多轮 RL（Multi-turn with Environment Feedback）
#
#  执行流程：
#    1. 模型输出第一轮推理（含四个步骤）
#    2. _run_env 提取 EDIT 中的 SMILES，用 RDKit 计算真实属性
#    3. 属性结果作为新的 user message 注入对话历史
#    4. 模型基于反馈输出第二轮（精细化修改或验证）
#    5. 重复至达标或达到 max_turns
#    6. 用折扣回报（γ-discounted return）聚合各轮奖励，构造 Datum
#
#  折扣回报的意义：
#    早期步骤的行为对最终结果有长程影响，折扣系数 γ 在奖励中体现
#    这种时间依赖关系。γ 越大（→1），越鼓励长期规划；越小，越
#    关注即时收益。对于分子优化，推荐 γ=0.9。
#
#  与模式 A 的差异总结：
#    相同：score_step、aggregate_step_rewards、权重配置
#    不同：_run_env 在步骤间被真实调用；奖励通过折扣回报而非即时值分配
# ══════════════════════════════════════════════════════════════════════════════

def run_mode_b(
    example: dict,
    renderer,
    sampling_client,
    tokenizer,
    ema_baseline: EMABaseline,
    num_trajectories: int = 2,   # K：每个 prompt 采多少条独立轨迹
    max_turns: int = 3,
    gamma: float = 0.9,
    max_tokens: int = 256,
    temperature: float = 0.7,
    early_stop_threshold: float = 0.7,
    debug: bool = False,
    debug_prefix: str = "",
) -> tuple[list[types.Datum], list[float], dict]:
    """
    修复后的 Mode B：
      1. 对同一 prompt 采 K 条完整轨迹（每条独立运行 multi-turn episode）
      2. 计算每条轨迹的折扣回报 G_k
      3. 组内归一化得到相对 advantage
      4. EMA baseline 做跨 prompt 的二次校正（可选）
      5. 为每条轨迹的每个 turn 构造 Datum
 
    返回值：
      datums       - 所有轨迹、所有 turn 的 Datum 列表
      traj_returns - 每条轨迹的折扣回报 G_k（用于日志）
      details      - 各子项均值（用于监控）
    """
    prompt_text = str(example[PROMPT_COL])
    context = {"prompt_text": prompt_text, "mol_in": _mol_from_prompt(prompt_text)}

    # 每个 prompt 组重置重复惩罚计数器，避免跨组污染奖励尺度
    if "_completion_counter" in globals():
        _completion_counter.clear()
    if "_fragment_counter" in globals():
        _fragment_counter.clear()

    all_trajectories = []

    for traj_idx in range(num_trajectories):
        messages = make_prompt_messages(example, system_prompt=SYSTEM_PROMPT_MULTISTEP)
        turns    = []
 
        for turn_idx in range(max_turns):
            prompt_input = renderer.build_generation_prompt(messages)
            prompt_tokens = prompt_input.to_ints()
 
            result = sampling_client.sample(
                prompt=prompt_input,
                sampling_params=types.SamplingParams(
                    max_tokens=max_tokens,
                    temperature=temperature,
                    stop=renderer.get_stop_sequences(),
                ),
                num_samples=1,
            ).result()

            seq = result.sequences[0]
            completion_text = tokenizer.decode(
                seq.tokens,
                skip_special_tokens=True,
            ).replace("<|im_end|>", "").strip()
            steps = parse_reasoning_steps(completion_text)

            step_rewards = {tag: score_step(tag, text, context) for tag, text in steps.items()}
            process_r = aggregate_step_rewards(step_rewards)
            turn_r = process_r
            outcome_r = None

            env_feedback, mol_out = _run_env(steps.get("EDIT", ""), prompt_text)
 
            # 最后一轮叠加 outcome reward
            is_last = (turn_idx == max_turns - 1) or (
                mol_out and process_r > early_stop_threshold
            )
            if is_last:
                edit_text         = steps.get("EDIT", "")
                outcome_scores, _ = compute_total_reward(
                    [edit_text], prompts=[prompt_text]
                )
                turn_r += OUTCOME_WEIGHT * outcome_scores[0]
 
            turns.append({
                "prompt_tokens": prompt_tokens,
                "comp_tokens":   list(seq.tokens),
                "comp_logprobs": list(seq.logprobs),
                "turn_reward":   turn_r,
            })

            if debug:
                print(
                    f"[DBG]{debug_prefix} traj={traj_idx} turn={turn_idx} "
                    f"step={step_rewards} process={process_r:.4f} "
                    f"outcome={outcome_r if outcome_r is not None else 'NA'} "
                    f"turn_reward={turn_r:.4f} is_last={is_last}"
                )

            messages.append({"role": "assistant", "content": completion_text})
            messages.append({"role": "user",      "content": env_feedback})
 
            if is_last:
                break
 
        # ── 折扣回报：从最后一轮向前递推 ──
        n       = len(turns)
        returns = [0.0] * n
        running = 0.0
        for i in reversed(range(n)):
            running    = turns[i]["turn_reward"] + gamma * running
            returns[i] = running
 
        # G_k = 第一轮的折扣回报（代表整条轨迹的价值）
        G_k = returns[0] if returns else 0.0
        all_trajectories.append((turns, returns, G_k))

        if debug:
            turn_rewards_dbg = [round(t["turn_reward"], 4) for t in turns]
            returns_dbg = [round(x, 4) for x in returns]
            print(
                f"[DBG]{debug_prefix} traj={traj_idx} turn_rewards={turn_rewards_dbg} "
                f"returns={returns_dbg} G_k={G_k:.4f}"
            )

    G_values = [t[2] for t in all_trajectories]
    G_arr    = np.array(G_values, dtype=np.float64)
    G_mean   = G_arr.mean()
    G_std    = G_arr.std()
 
    if G_std < 1e-8:
        baselines = [ema_baseline.update_and_get(G) for G in G_values]
        relative_advs = [G - b for G, b in zip(G_values, baselines)]
        if debug:
            print(
                f"[DBG]{debug_prefix} G_values={G_values} G_std≈0 baselines={baselines} "
                f"relative_advs={relative_advs}"
            )
    else:
        # 正常情况：组归一化给排序，EMA baseline 给尺度校正（可选）
        group_normalized = ((G_arr - G_mean) / G_std).tolist()
        baselines = [ema_baseline.update_and_get(g) for g in group_normalized]
        relative_advs = [g - b for g, b in zip(group_normalized, baselines)]
        if debug:
            print(
                f"[DBG]{debug_prefix} G_values={G_values} mean={G_mean:.4f} std={G_std:.4f} "
                f"group_norm={group_normalized} baselines={baselines} relative_advs={relative_advs}"
            )

    datums = []
    traj_returns = []
    dropped_zero_adv = 0

    for idx, ((turns, returns, G_k), adv) in enumerate(zip(all_trajectories, relative_advs)):
        traj_returns.append(G_k)

        if abs(adv) < 1e-12:
            dropped_zero_adv += 1
            if debug:
                print(f"[DBG]{debug_prefix} drop traj={idx} because adv≈0 ({adv:.4e})")
            continue

        for turn, turn_G in zip(turns, returns):
            # turn 级 advantage = 轨迹级归一化 adv × (turn_G / G_k) 比例修正
            # 当 G_k ≈ 0 时退化为直接用 adv
            turn_scale = (turn_G / G_k) if abs(G_k) > 1e-8 else 1.0
            turn_adv   = adv * turn_scale
 
            n_p     = len(turn["prompt_tokens"])
            n_c     = len(turn["comp_tokens"])
            full    = turn["prompt_tokens"] + turn["comp_tokens"]
            min_len = min(
                len(full) - 1,
                n_p - 1 + n_c,
                n_p - 1 + len(turn["comp_logprobs"]),
            )
 
            datums.append(types.Datum(
                model_input=types.ModelInput.from_ints(tokens=full[:-1]),
                loss_fn_inputs=dict(
                    target_tokens=full[1:][:min_len],
                    advantages=([0.0] * (n_p - 1) + [turn_adv] * n_c)[:min_len],
                    logprobs=([0.0] * (n_p - 1) + turn["comp_logprobs"])[:min_len],
                ),
            ))
 
    details = {
        "mean_G":   float(G_mean),
        "std_G":    float(G_std),
        "mean_adv": float(np.mean(relative_advs)) if relative_advs else 0.0,
        "num_traj": len(all_trajectories),
        "num_turns_total": sum(len(t[0]) for t in all_trajectories),
        "dropped_zero_adv": dropped_zero_adv,
    }
    return datums, traj_returns, details



In [24]:
# Create a NEW training client for GRPO, starting from the SFT checkpoint
grpo_training_client = service_client.create_lora_training_client(
    base_model=BASE_MODEL,
    rank=LORA_RANK,
)

# # Load the SFT weights as our starting point
# grpo_training_client.load_state("tinker://88c08ced-d3ee-5776-b214-1e7ef8063c76:train:0/weights/MultistepRL-sft-final.weekend.Qwen8B")
# print("Loaded SFT checkpoint into GRPO training client")

# Create initial sampling client
grpo_sampling_client = grpo_training_client.save_weights_and_get_sampling_client(
    name="grpo-init"
)
print("GRPO sampling client ready")

GRPO sampling client ready


In [11]:
# ══════════════════════════════════════════════════════════════════════════════
#  修复6：与多步输出格式兼容的评估函数
#
#  原始问题：
#    原 evaluate_with_new_reward 把整条 completion 传入 compute_total_reward，
#    而 strict_format_reward 期望的格式是：
#      "Replace X with Y to form <SMILES>...</SMILES>."
#    多步输出实际格式是：
#      "[ANALYZE] ... [PLAN] ... [EDIT] Replace X with Y to form <SMILES>...</SMILES> [VERIFY] ..."
#
#    后果：
#      ① strict_format_reward 对多步输出几乎永远返回 -4.0（格式不匹配）
#      ② SMILES 可能从 ANALYZE/VERIFY 步骤中错误提取（第一个出现的 <SMILES>）
#      ③ property_reward 拿错 SMILES，计算出的属性值完全不对
#
#  修复策略：
#    ① 先用 parse_reasoning_steps 提取 EDIT 步骤
#    ② 对 EDIT 步骤单独运行 strict_format_reward（格式检查）
#    ③ 对 EDIT 步骤提取 SMILES，再运行 property reward
#    ④ 新增"步骤完整性"指标：四步是否都存在
#    ⑤ 分离展示"过程质量"和"结果质量"两个维度
# ══════════════════════════════════════════════════════════════════════════════
 
from rdkit.Chem import Draw
from IPython.display import display

def evaluate_multistep(
    sampling_client,
    prompt_input,
    prompt_text: str,
    tokenizer,
    renderer,
    temperature: float = 0.0,
    max_tokens: int = 512,
) -> dict:
    """
    兼容多步输出格式的评估函数，替换原始的 evaluate_with_new_reward。
 
    返回字典包含：
      text            - 完整 completion 文本
      steps           - 解析出的四个步骤文本
      step_presence   - 各步骤是否存在 {ANALYZE: bool, ...}
      completeness    - 存在步骤数 / 4（0~1）
      smiles          - 从 EDIT 步骤提取的 SMILES（而非全文第一个）
      valid_smiles    - SMILES 是否合法
      strict          - EDIT 步骤的格式得分
      property        - 属性方向匹配得分（基于 EDIT SMILES）
      r_analyze       - ANALYZE 步骤得分
      r_plan          - PLAN 步骤得分
      r_edit          - EDIT 步骤得分（process reward）
      r_verify        - VERIFY 步骤得分
      process_total   - 四步 process reward 加权和
      outcome_total   - strict + property 得分（最终化学质量）
      combined        - process + outcome 的组合分（与训练 reward 对齐）
    """
    result = sampling_client.sample(
        prompt=prompt_input,
        sampling_params=types.SamplingParams(
            max_tokens=max_tokens,
            temperature=temperature,
            stop=renderer.get_stop_sequences(),
        ),
        num_samples=1,
    ).result()

    completion_text = tokenizer.decode(
        result.sequences[0].tokens,
        skip_special_tokens=True,
    ).replace("<|im_end|>", "").strip()

    steps = parse_reasoning_steps(completion_text)
    step_presence = {tag: bool(text) for tag, text in steps.items()}
    completeness  = sum(step_presence.values()) / 4.0
 
    # ── 从 EDIT 步骤提取 SMILES（修复点②）──
    edit_text  = steps.get("EDIT", "")
    smiles_m   = re.search(r"<SMILES>(.*?)</SMILES>", edit_text)
    pred_smiles = smiles_m.group(1).strip() if smiles_m else None
    valid_smiles = bool(pred_smiles and Chem.MolFromSmiles(pred_smiles))
 
    # ── 格式得分：只对 EDIT 步骤打分（修复点②③）──
    strict_scores = strict_format_reward([edit_text])
    strict_score  = float(strict_scores[0]) if strict_scores else -4.0
 
    # ── 属性得分：使用 EDIT 步骤的 SMILES（修复点③）──
    # 将 EDIT 步骤单独传入，保证 property reward 拿到正确的 SMILES
    property_scores = multi_property_reward([edit_text], prompts=[prompt_text])
    property_score  = float(property_scores[0]) if property_scores else -4.0
 
    # ── Process reward（与训练时完全一致）──
    context = {"prompt_text": prompt_text, "mol_in": _mol_from_prompt(prompt_text)}
    step_scores = {tag: score_step(tag, text, context) for tag, text in steps.items()}
    process_total = aggregate_step_rewards(step_scores)
 
    # ── 组合分（与训练 reward 公式对齐）──
    outcome_total = (
        REWARD_WEIGHTS["strict"]   * strict_score +
        REWARD_WEIGHTS["property"] * property_score
    )
    combined = process_total + OUTCOME_WEIGHT * outcome_total
 
    return {
        "text":          completion_text,
        "steps":         steps,
        "step_presence": step_presence,
        "completeness":  completeness,
        "smiles":        pred_smiles,
        "valid_smiles":  valid_smiles,
        "strict":        strict_score,
        "property":      property_score,
        "r_analyze":     step_scores.get("ANALYZE", 0.0),
        "r_plan":        step_scores.get("PLAN",    0.0),
        "r_edit":        step_scores.get("EDIT",    0.0),
        "r_verify":      step_scores.get("VERIFY",  0.0),
        "process_total": process_total,
        "outcome_total": outcome_total,
        "combined":      combined,
    }
 
 
def run_evaluation(
    sft_sampling_client,
    grpo_sampling_client,
    raw_data,
    renderer,
    tokenizer,
    num_examples: int = 5,
    rng_seed: int = 42,
):
    """对比 SFT 和 GRPO 模型（多步兼容版本）。"""
    from rdkit.Chem import Draw
    from IPython.display import display

    rng = np.random.default_rng(rng_seed)
    eval_indices = rng.choice(len(raw_data), size=min(num_examples, len(raw_data)), replace=False)
    stop_seqs    = renderer.get_stop_sequences()
 
    # 聚合指标
    metrics = {"sft": defaultdict(list), "grpo": defaultdict(list)}
 
    print("=" * 100)
    print(f"{'指标':<20} {'SFT':>10} {'GRPO':>10}   说明")
    print("=" * 100)
 
    for idx in eval_indices:
        example      = raw_data[int(idx)]
        prompt_text  = str(example[PROMPT_COL])
        ground_truth = str(example[ANSWER_COL])
 
        prompt_msgs  = make_prompt_messages(example, system_prompt=SYSTEM_PROMPT_MULTISTEP)
        prompt_input = renderer.build_generation_prompt(prompt_msgs)
 
        sft_eval  = evaluate_multistep(sft_sampling_client,  prompt_input, prompt_text, tokenizer, renderer)
        grpo_eval = evaluate_multistep(grpo_sampling_client, prompt_input, prompt_text, tokenizer, renderer)
 
        for name, ev in [("sft", sft_eval), ("grpo", grpo_eval)]:
            metrics[name]["completeness"].append(ev["completeness"])
            metrics[name]["valid_smiles"].append(float(ev["valid_smiles"]))
            metrics[name]["strict"].append(ev["strict"])
            metrics[name]["property"].append(ev["property"])
            metrics[name]["r_edit"].append(ev["r_edit"])
            metrics[name]["process_total"].append(ev["process_total"])
            metrics[name]["combined"].append(ev["combined"])
 
        # 展示单条结果
        print(f"\nPrompt: {prompt_text[:100]}...")
        print(f"Ground Truth SMILES: {_extract_smiles_from_edit(ground_truth)}")
        print()
 
        for name, ev in [("SFT", sft_eval), ("GRPO", grpo_eval)]:
            presence_str = " ".join(
                f"[{t}]{'✓' if ev['step_presence'][t] else '✗'}"
                for t in ["ANALYZE", "PLAN", "EDIT", "VERIFY"]
            )
            print(f"  {name}  步骤: {presence_str}")
            print(f"       SMILES: {ev['smiles']} {'✓' if ev['valid_smiles'] else '✗'}")
            print(f"       过程: analyze={ev['r_analyze']:.2f} plan={ev['r_plan']:.2f} "
                  f"edit={ev['r_edit']:.2f} verify={ev['r_verify']:.2f} → {ev['process_total']:.3f}")
            print(f"       结果: strict={ev['strict']:.2f} property={ev['property']:.3f} → {ev['outcome_total']:.3f}")
            print(f"       综合: {ev['combined']:.4f}")
 
        # 可视化
        gt_smiles  = _extract_smiles_from_edit(ground_truth) or ground_truth.strip()
        mols, legs = [], []
        for smi, label in [(gt_smiles, "Ground Truth"),
                           (sft_eval["smiles"], "SFT"),
                           (grpo_eval["smiles"], "GRPO")]:
            if smi:
                mol = Chem.MolFromSmiles(smi)
                if mol:
                    mols.append(mol)
                    legs.append(label)
        if mols:
            img = Draw.MolsToGridImage(mols, legends=legs, molsPerRow=3, subImgSize=(300, 250))
            display(img)
 
    # 汇总表格
    print("\n" + "=" * 100)
    print("汇总（均值）")
    print(f"{'指标':<25} {'SFT':>10} {'GRPO':>10}   说明")
    print("-" * 75)
 
    report_items = [
        ("completeness",  "步骤完整率（0~1）"),
        ("valid_smiles",  "合法 SMILES 率"),
        ("strict",        "格式得分（EDIT步骤）"),
        ("property",      "属性方向匹配得分"),
        ("r_edit",        "EDIT process reward"),
        ("process_total", "过程 reward 合计"),
        ("combined",      "综合得分（训练对齐）"),
    ]
    for key, desc in report_items:
        s = np.mean(metrics["sft"][key])
        g = np.mean(metrics["grpo"][key])
        arrow = "↑" if g > s else ("↓" if g < s else "=")
        print(f"  {desc:<23} {s:>10.4f} {g:>10.4f}  {arrow}")
 
    print("=" * 100)
 
 
def _extract_smiles_from_edit(text: str) -> Optional[str]:
    """
    从包含 [EDIT] 标签的文本中提取 SMILES。
    若文本无 [EDIT] 标签，退而从全文找第一个 <SMILES>。
    """
    steps = parse_reasoning_steps(text)
    edit  = steps.get("EDIT", "")
    m     = re.search(r"<SMILES>(.*?)</SMILES>", edit)
    if m:
        return m.group(1).strip()
    # 回退：尝试全文
    m2 = re.search(r"<SMILES>(.*?)</SMILES>", text)
    return m2.group(1).strip() if m2 else None
 
 

In [12]:
# ══════════════════════════════════════════════════════════════════════════════
#  第六部分：统一训练循环
#
#  通过 mode 参数选择执行方式，其余逻辑（PPO 更新、日志、checkpoint）
#  完全共用，便于对照实验。
# ══════════════════════════════════════════════════════════════════════════════



# ══════════════════════════════════════════════════════════════════════════════
#  修复后的完整训练循环（整合修复5和修复6的监控）
# ══════════════════════════════════════════════════════════════════════════════
 
def run_multistep_rl_training_fixed(
    raw_data,
    renderer,
    training_client,
    tokenizer,
    mode: str = "B",
    num_iterations: int = 50,
    prompts_per_iter: int = 8,
    learning_rate: float = 3e-6,
    # Mode A 参数
    samples_per_prompt: int = 4,
    # Mode B 参数（修复5）
    trajectories_per_prompt: int = 2,   # K，每个 prompt 采多少条轨迹
    max_turns: int = 3,
    gamma: float = 0.9,
    ema_alpha: float = 0.05,
    rng_seed: int = 42,
    debug_adv: bool = False,
    debug_prompts_per_iter: int = 1,
    debug_iterations: int = 1,
    keep_skipped_metrics: bool = True,
):
    """
    修复后的统一训练循环。
 
    修复5的关键改动：
      - Mode B 新增 trajectories_per_prompt 参数（K=2~3）
      - 全局 EMABaseline 实例跨 iteration 持久化
      - run_mode_b_fixed 替换原 run_mode_b
 
    修复6 在此处体现为训练日志新增了 std_G（轨迹回报标准差）监控，
    若 std_G 长期接近 0，说明模型已收敛或陷入局部最优。
    """
    assert mode in ("A", "B")
    rng      = np.random.default_rng(rng_seed)
    ema      = EMABaseline(alpha=ema_alpha)   # 修复5：跨 iteration 持久化
    sampling_client = training_client.save_weights_and_get_sampling_client(
        name=f"multistep-{mode}-init"
    )
 
    all_metrics = []
    print(f"启动修复后的多步 RL 训练（模式 {mode}）")
 
    if mode == "A":
        header = f"{'Iter':>4} | {'Mean R':>8} | {'r_edit':>7} | {'outcome':>8} | {'#data':>6} | {'t(s)':>6}"
    else:
        header = f"{'Iter':>4} | {'Mean G':>8} | {'std_G':>7} | {'mean_adv':>9} | {'#data':>6} | {'t(s)':>6}"
    print(header)
    print("-" * len(header))
 
    for iteration in range(num_iterations):
        import time
        t0 = time.time()

        prompt_indices = rng.choice(len(raw_data), size=prompts_per_iter, replace=False)
        all_datums     = []
        all_rewards    = []
        all_details    = []
 
        for local_idx, prompt_idx in enumerate(prompt_indices):
            example = raw_data[int(prompt_idx)]
            try:
                if mode == "A":
                    datums, rewards, details = run_mode_a(
                        example, renderer, sampling_client, tokenizer,
                        num_samples=samples_per_prompt,
                    )
                else:
                    do_debug = (
                        debug_adv
                        and iteration < debug_iterations
                        and local_idx < debug_prompts_per_iter
                    )
                    datums, rewards, details = run_mode_b(
                        example, renderer, sampling_client, tokenizer,
                        ema_baseline=ema,
                        num_trajectories=trajectories_per_prompt,
                        max_turns=max_turns,
                        gamma=gamma,
                        debug=do_debug,
                        debug_prefix=f" it={iteration} p={int(prompt_idx)} ",
                    )
            except Exception as e:
                print(f"  [跳过] prompt {prompt_idx}: {e}")
                continue
 
            all_datums.extend(datums)
            all_rewards.extend(rewards)
            all_details.append(details)
 
        elapsed = time.time() - t0

        if not all_datums:
            if mode == "B" and all_details:
                mean_G = float(np.mean([d.get("mean_G", 0.0) for d in all_details]))
                std_G = float(np.mean([d.get("std_G", 0.0) for d in all_details]))
                mean_adv = float(np.mean([d.get("mean_adv", 0.0) for d in all_details]))
                dropped = int(np.sum([d.get("dropped_zero_adv", 0) for d in all_details]))
                print(
                    f"{iteration:>4} | 无有效数据，跳过 "
                    f"(原因: adv≈0 导致轨迹被过滤; mean_G={mean_G:.4f}, std_G={std_G:.4f}, mean_adv={mean_adv:.4f}, dropped={dropped})"
                )
                if keep_skipped_metrics:
                    all_metrics.append({
                        "iteration": iteration,
                        "mode": mode,
                        "mean_G": mean_G,
                        "std_G": std_G,
                        "mean_adv": mean_adv,
                        "num_data": 0,
                        "skipped": True,
                        "dropped_zero_adv": dropped,
                        "time": elapsed,
                    })
            else:
                print(f"{iteration:>4} | 无有效数据，跳过")
                if keep_skipped_metrics:
                    all_metrics.append({
                        "iteration": iteration,
                        "mode": mode,
                        "num_data": 0,
                        "skipped": True,
                        "time": elapsed,
                    })
            continue
 
        # PPO 更新（两种模式相同）
        fwdbwd = training_client.forward_backward(
            all_datums, loss_fn="ppo",
            loss_fn_config={"clip_low_threshold": 0.9, "clip_high_threshold": 1.1},
        )
        optim = training_client.optim_step(types.AdamParams(learning_rate=learning_rate))
        fwdbwd.result()
        optim.result()
 
        sampling_client = training_client.save_weights_and_get_sampling_client(
            name=f"multistep-{mode}-iter-{iteration:04d}"
        )
 
        # 日志
        if mode == "A":
            mean_r   = float(np.mean(all_rewards))
            mean_edit = float(np.mean([d.get("EDIT", 0) for d in all_details]))
            mean_out  = float(np.mean([d.get("outcome", 0) for d in all_details]))
            print(f"{iteration:>4} | {mean_r:>8.4f} | {mean_edit:>7.3f} | {mean_out:>8.4f} | {len(all_datums):>6} | {elapsed:>6.1f}")
            metric = dict(iteration=iteration, mode=mode, mean_reward=mean_r,
                          mean_edit=mean_edit, mean_outcome=mean_out,
                          num_data=len(all_datums), time=elapsed, skipped=False)
        else:
            # 修复5新增：监控 std_G（判断多样性）和 mean_adv（判断学习方向）
            mean_G   = float(np.mean([d.get("mean_G",   0) for d in all_details]))
            std_G    = float(np.mean([d.get("std_G",    0) for d in all_details]))
            mean_adv = float(np.mean([d.get("mean_adv", 0) for d in all_details]))
            dropped = int(np.sum([d.get("dropped_zero_adv", 0) for d in all_details]))
            print(f"{iteration:>4} | {mean_G:>8.4f} | {std_G:>7.4f} | {mean_adv:>9.4f} | {len(all_datums):>6} | {elapsed:>6.1f}")
            metric = dict(iteration=iteration, mode=mode, mean_G=mean_G, std_G=std_G,
                          mean_adv=mean_adv, num_data=len(all_datums),
                          dropped_zero_adv=dropped,
                          ema_baseline=ema.value, time=elapsed, skipped=False)
 
        all_metrics.append(metric)
 
    print(f"\n训练完成（模式 {mode}，{num_iterations} 轮）")
    globals()["all_metrics"] = all_metrics
    return all_metrics

In [ ]:
# Run multi-step RL training (set mode="A" for single-round PRM, mode="B" for multi-turn RL)
all_metrics = run_multistep_rl_training_fixed(
    raw_data=raw_data,
    renderer=renderer,
    training_client=grpo_training_client,
    tokenizer=tokenizer,
    mode="B",
    num_iterations=2,
    prompts_per_iter=1,
    learning_rate=3e-6,
    trajectories_per_prompt=4,
    max_turns=3,
    gamma=0.9,
    ema_alpha=0.05,
    rng_seed=42,
    debug_adv=True,
    debug_prompts_per_iter=1,
    debug_iterations=3,
    keep_skipped_metrics=True,
 )

print(f"Training finished. Collected {len(all_metrics)} metric points.")

## 8. Plot Training Curves

In [ ]:
from matplotlib import pyplot as plt

if "all_metrics" not in globals():
    raise ValueError(
        "all_metrics is undefined. Run: all_metrics = run_multistep_rl_training_fixed(...) first."
    )
if not all_metrics:
    raise ValueError("all_metrics is empty. Please run training first.")

iters = [m["iteration"] for m in all_metrics]
first = all_metrics[0]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

if "mean_reward" in first:
    # Mode A metrics
    mean_reward = [m.get("mean_reward", 0.0) for m in all_metrics]
    mean_edit = [m.get("mean_edit", 0.0) for m in all_metrics]
    mean_outcome = [m.get("mean_outcome", 0.0) for m in all_metrics]

    axes[0].plot(iters, mean_reward, "b-o", markersize=3)
    axes[0].set_title("Mean Reward")
    axes[0].set_ylabel("Reward")

    axes[1].plot(iters, mean_edit, "g-o", markersize=3)
    axes[1].set_title("Mean EDIT Reward")
    axes[1].set_ylabel("Reward")

    axes[2].plot(iters, mean_outcome, "r-o", markersize=3)
    axes[2].set_title("Mean Outcome Reward")
    axes[2].set_ylabel("Reward")

elif "mean_G" in first:
    # Mode B metrics
    mean_G = [m.get("mean_G", 0.0) for m in all_metrics]
    std_G = [m.get("std_G", 0.0) for m in all_metrics]
    mean_adv = [m.get("mean_adv", 0.0) for m in all_metrics]

    axes[0].plot(iters, mean_G, "b-o", markersize=3)
    axes[0].set_title("Mean Trajectory Return (G)")
    axes[0].set_ylabel("Return")

    axes[1].plot(iters, std_G, "g-o", markersize=3)
    axes[1].set_title("Std of Trajectory Return")
    axes[1].set_ylabel("Std")

    axes[2].plot(iters, mean_adv, "r-o", markersize=3)
    axes[2].set_title("Mean Advantage")
    axes[2].set_ylabel("Advantage")

else:
    raise ValueError("Unknown metric schema in all_metrics.")

for ax in axes:
    ax.set_xlabel("Iteration")
    ax.grid(True, alpha=0.3)

plt.suptitle("Multi-step RL Training Progress", fontsize=14)
plt.tight_layout()
plt.show()

## 9. Evaluate the Final Model

In [ ]:
sft_training_client = service_client.create_lora_training_client(
    base_model=BASE_MODEL,
    rank=LORA_RANK,
)
sft_training_client.load_state("tinker://74e5f0bc-93c9-5df1-9570-5ed0910d9daa:train:0/weights/sft-final")
sft_sampling_client = sft_training_client.save_weights_and_get_sampling_client(name="sft-sampler.weekend")
grpo_training_client = service_client.create_lora_training_client(
    base_model=BASE_MODEL,
    rank=LORA_RANK,
)
grpo_training_client.load_state("tinker://eb393fba-89e6-5005-b2f2-3df09d6142c3:train:0/weights/grpo-final.weekend")
grpo_sampling_client = grpo_training_client.save_weights_and_get_sampling_client(name="grpo-sampler.weekend")

In [ ]:
# 使用多步兼容评估函数，避免旧版评估对 [ANALYZE]/[PLAN]/[EDIT]/[VERIFY] 格式误判
run_evaluation(
    sft_sampling_client=sft_sampling_client,
    grpo_sampling_client=grpo_sampling_client,
    raw_data=raw_data,
    renderer=renderer,
    tokenizer=tokenizer,
    num_examples=5,
    rng_seed=42,
)

In [ ]:
# Save the final GRPO checkpoint
grpo_save_result = grpo_training_client.save_state(name="MultistepRL-grpo-final.weekend.Qwen8B").result()
grpo_checkpoint_path = grpo_save_result.path
print(f"GRPO checkpoint saved at: {grpo_checkpoint_path}")

## 10. Download the Trained Weights

You can download the LoRA adapter weights to use with your own inference setup (e.g., vLLM, SGLang, or HuggingFace).

In [ ]:
# Download the GRPO model weights
# The downloaded weights are a LoRA adapter that can be loaded
# with HuggingFace PEFT or merged into the base model.

rest_client = service_client.create_rest_client()

# List available checkpoints
# rest_client.list_checkpoints()  # Uncomment to see all checkpoints

# Download to a local directory
# Uncomment and run when ready:
# rest_client.download_checkpoint(grpo_checkpoint_path, output_dir="./grpo-weights")
# print("Weights downloaded to ./grpo-weights/")

print(f"GRPO checkpoint path: {grpo_checkpoint_path}")
print("Uncomment the download line above when you're ready to export.")

---

## Summary & Next Steps

In this notebook, we:

1. **Loaded** the `Alan123/homo-lumo-grpo` dataset from HuggingFace
2. **SFT-trained** a Qwen3 model to learn the chemistry Q&A format using `cross_entropy` loss
3. **GRPO-trained** the model using RL with a custom reward function and `ppo` loss
4. **Compared** SFT vs GRPO predictions side by side

### Tips for better results:

- **More SFT steps** — if your dataset is large, train for more steps or multiple epochs
- **Tune the reward function** — the reward function is the most important part of GRPO. Make it reflect what you actually care about.
- **Try different loss functions** — Tinker supports `importance_sampling`, `ppo`, `cispo`, and `dro`. See the [Loss Functions docs](https://tinker-docs.thinkingmachines.ai/losses)
- **Increase samples per prompt** — more completions per prompt = better advantage estimates
- **Use a larger model** — `Qwen/Qwen3-30B-A3B` (MoE) is very cost-effective and much more capable
- **Use the Tinker Cookbook** — the cookbook has optimized training loops with async operations, logging, and eval. See [rl/train.py](https://github.com/thinking-machines-lab/tinker-cookbook/blob/main/tinker_cookbook/rl/train.py)

### Key Tinker resources:
- [Tinker API Docs](https://tinker-docs.thinkingmachines.ai/)
- [Supervised Learning Guide](https://tinker-docs.thinkingmachines.ai/supervised-learning/sl-basic)
- [RL Training Guide](https://tinker-docs.thinkingmachines.ai/rl/rl-basic)
- [Model Lineup](https://tinker-docs.thinkingmachines.ai/model-lineup)
- [Tinker Cookbook (GitHub)](https://github.com/thinking-machines-lab/tinker-cookbook)

In [27]:
# 诊断（精简版）：查看指定 prompt 的逐轮原始输出与四步解析内容
prompt_idx = 14584
max_turns = 3

example = raw_data[int(prompt_idx)]
prompt_text = str(example[PROMPT_COL])
ground_truth = str(example[ANSWER_COL])

mol_in = _mol_from_prompt(prompt_text)
qed_val  = QED.qed(mol_in)
logp_val = Crippen.MolLogP(mol_in)
mw_val   = Descriptors.ExactMolWt(mol_in)

print("=" * 100)
print(f"Prompt idx={prompt_idx}")
print(prompt_text[:])
print(ground_truth[:])
print(f"Input molecule: QED={qed_val:.4f} logP={logp_val:.4f} MW={mw_val:.2f}")
print("=" * 100)

messages = make_prompt_messages(example, system_prompt=SYSTEM_PROMPT_MULTISTEP)

for turn_idx in range(max_turns):
    prompt_input = renderer.build_generation_prompt(messages)
    result = grpo_sampling_client.sample(
        prompt=prompt_input,
        sampling_params=types.SamplingParams(
            max_tokens=256,
            temperature=0.7,
            stop=renderer.get_stop_sequences(),
        ),
        num_samples=1,
    ).result()

    seq = result.sequences[0]
    completion_text = tokenizer.decode(
        seq.tokens,
        skip_special_tokens=True,
    ).replace("<|im_end|>", "").strip()

    steps = parse_reasoning_steps(completion_text)
    context = {"prompt_text": prompt_text, "mol_in": _mol_from_prompt(prompt_text)}

    step_rewards = {tag: score_step(tag, text, context) for tag, text in steps.items()}

    print(f"\n\n----- TURN {turn_idx} -----")
    print("Raw completion (first 800 chars):")
    print(completion_text[:800])

    for tag in ["ANALYZE", "PLAN", "EDIT", "VERIFY"]:
        txt = steps.get(tag, "")
        print(f"\n[{tag}] reward={step_rewards.get(tag, 0.0):.4f} present={bool(txt)} len={len(txt)}")
        print(txt if txt else "<EMPTY>")

    env_feedback, mol_out = _run_env(steps.get("EDIT", ""), prompt_text)
    process_r = aggregate_step_rewards(step_rewards)
    is_last = (turn_idx == max_turns - 1) or ((mol_out is not None) and process_r > 0.7)

    print("\nEnv feedback:")
    print(env_feedback)
    print(f"is_last={is_last}")

    messages.append({"role": "assistant", "content": completion_text})
    messages.append({"role": "user", "content": env_feedback})

    if is_last:
        break

Prompt idx=14584
Given the intermediate molecule SMILES <SMILES>CC[C@H]1CC[C@](O)([C@]2(C#N)CCc3ccccc32)C1</SMILES>, which is composed of fragments ['CC', '[C@H]1CC[C@](O)([C@]2(C#N)CCc3ccccc32)C1']. Propose a single replace, add or remove step on fragment level that makes the new molecule's QED <QED>0.079</QED> lower, LogP <LogP>0.350</LogP> lower, and Molecular Weight <MW>31.076</MW> higher.
Replace [C@H]1CC[C@](O)([C@]2(C#N)CCc3ccccc32)C1 with [C@H]1CCC(=O)[C@H]([C@@H]2CCC[C@H](S(C)(=O)=O)C2)C1 to form <SMILES>CC[C@H]1CCC(=O)[C@H]([C@@H]2CCC[C@H](S(C)(=O)=O)C2)C1</SMILES>.
Input molecule: QED=0.8805 logP=3.3354 MW=255.16


----- TURN 0 -----
Raw completion (first 800 chars):
<think>
</think>

[ANALYZE] Current molecule: QED=0.65, LogP=4.2, MW=380. Target: QED lower, LogP lower, MW higher.  
[PLAN] Introduce a hydroxyl group on the cyclopropane ring to increase MW and slightly decrease LogP, while reducing QED through increased polarity.  
[EDIT] Replace [C@H]1CC[C@](O)([C@]2(C#N)CCc